# Histocompatibility and Immunogenetics — HL7 v2 to and from the NW Standard

This is the tenth notebook in the series. Unlike `01`–`09`, this one stays entirely in
**HL7 v2** — no FHIR anywhere — and hand-builds the field-level conversion a Trust
Integration Engine (TIE) has to do to turn its own Trust's local HL7 v2 flavour into
the shared **NW HL7 v2 standard**
([nw-gmsa.github.io/en/hl7v2.html](https://nw-gmsa.github.io/en/hl7v2.html)).

The worked example is the
[Histocompatibility and Immunogenetics (H&I) use case](https://nw-gmsa.github.io/en/HistocompatibilityAndImmunogenetics.html):
moving The Clatterbridge Cancer Centre's existing order/report feed — Meditech EPR
direct to a local lab — onto a regionally-hosted LIMS, Histotrac. The same pattern
extends to StarLIMS and iGene, NW Genomics' other regional LIMS.

- `Input/V2/O01/Clatterbridge-Order.txt` — an order as it reaches the local lab today,
  in Clatterbridge's own Meditech-originated flavour. This notebook's worked example.
- `Input/V2/O01/histotrac-MFT.txt` — a real order from a different Trust (MFT),
  already in the shape the RIE hands on to Histotrac — used below to check one of this
  notebook's own conversion decisions against a real-world example.
- `Input/V2/R01/histotrac-MFT.txt` / `Input/V2/R01/histotrac.txt` — reports coming back
  the other way, Histotrac → RIE → Trust.


## What the NW HL7 v2 standard is built from

[nw-gmsa.github.io/en/hl7v2.html](https://nw-gmsa.github.io/en/hl7v2.html) isn't a
standard NW Genomics invented from a blank page — it's a **union** of segment
definitions drawn from other national HL7 v2 profiles, plus NHS England's own data
standards:

- **NHS England's HL7 v2 ADT message specification** — `MSH`, `PID`, `PD1`, `PV1`, and
  the `XCN`/`XON` data types (person/organisation identifiers) are taken from here.
- **Digital Health and Care Wales (DHCW)'s HL7 v2 `ORU_R01` 2.5.1 Implementation
  Guide** — `ORC`, `OBR`, `OBX`, and `SPM` are taken from here.
- **NHS England's Data Model and Dictionary** and related national reference data (NHS
  Number, ODS codes, SNOMED CT, the England Genomic Test Directory) — used throughout
  both halves for identifiers and coded values, rather than each LIMS/Trust inventing
  its own.

Three message types are defined this way: `OML_O21` (laboratory order), `ORU_R01`
(laboratory report), and `MDM_T02` (document notification).

The H&I use case is a deliberate exception worth naming rather than glossing over: its
own page specifies `ORM_O01`/`ORU_R01`, and `hl7v2.html` defines no `ORM_O01` profile at
all — `OML_O21` is the only order message type it documents. In practice, both this
repo's fixtures and the live RIE treat `ORM^O01` at `MSH-12 = 2.5.1` as this use case's
version of "NW standard" — for H&I, the version alignment matters to the RIE more than
the trigger event choice. `IntegrationTest.py`'s `clatterbridge_histotrac` group
comment has the fuller history of how that got established (a real `AR` reject on
`MSH-12 = "2.3"`, fixed by bumping the version, not the trigger).

## The pattern: canonical data model + message routing

GitHub's notebook viewer doesn't render ```mermaid``` code fences (unlike GitHub's
regular Markdown-file viewer) — this diagram is embedded instead as an image from
[mermaid.ink](https://mermaid.ink) (base64-encoded diagram source in the URL itself, no
server-side state), which renders the same on GitHub, in Jupyter, and anywhere else that
can display an `<img>`.

![notebook 10 flow](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgICBzdWJncmFwaCBUUlVTVFsiTkhTIFRydXN0IChlLmcuIENsYXR0ZXJicmlkZ2UgLyBNRlQpIl0KICAgICAgICBFUFJbIkVQUlxuKE1lZGl0ZWNoIC8gRXBpYyBCZWFrZXIpIl0KICAgICAgICBUSUVbIlRydXN0IEludGVncmF0aW9uIEVuZ2luZSAoVElFKSJdCiAgICBlbmQKICAgIHN1YmdyYXBoIE5XR1siTlcgR2Vub21pY3MiXQogICAgICAgIFJJRVsiUmVnaW9uYWwgSW50ZWdyYXRpb24gRW5naW5lIChSSUUpXG5jYW5vbmljYWwgZGF0YSBtb2RlbCArIG1lc3NhZ2Ugcm91dGluZyJdCiAgICBlbmQKICAgIHN1YmdyYXBoIExJTVNHWyJSZWdpb25hbCBMSU1TIl0KICAgICAgICBISVNUT1RSQUNbIkhpc3RvdHJhYyJdCiAgICAgICAgU1RBUkxJTVNbIlN0YXJMSU1TIl0KICAgICAgICBJR0VORVsiaUdlbmUiXQogICAgZW5kCgogICAgRVBSIC0tPnwibG9jYWwtZmxhdm91ciB2MiBvcmRlciJ8IFRJRQogICAgVElFIC0tPnwiTlcgc3RhbmRhcmQgdjIgb3JkZXJcbihPUk1fTzAxIC8gT01MX08yMSwgMi41LjEpInwgUklFCiAgICBSSUUgLS0+fCJIaXN0b3RyYWMtZmxhdm91ciB2MiBvcmRlciJ8IEhJU1RPVFJBQwogICAgUklFIC0uLT58IlN0YXJMSU1TLWZsYXZvdXIgdjIgb3JkZXIifCBTVEFSTElNUwogICAgUklFIC0uLT58ImlHZW5lLWZsYXZvdXIgdjIgb3JkZXIifCBJR0VORQoKICAgIEhJU1RPVFJBQyAtLT58Ikhpc3RvdHJhYy1mbGF2b3VyIHYyIHJlcG9ydCJ8IFJJRQogICAgUklFIC0tPnwiTlcgc3RhbmRhcmQgdjIgcmVwb3J0XG4oT1JVX1IwMSwgMi41LjEpInwgVElFCiAgICBUSUUgLS0+fCJsb2NhbC1mbGF2b3VyIHYyIHJlcG9ydCJ8IEVQUgo=)

Conversion from EPR or LIMS into HL7 v2 in the first place is each NHS Trust's own
**Trust Integration Engine (TIE)**'s job, and each LIMS's own vendor-specific interface
on the other end. In between sits NW Genomics' **Regional Integration Engine (RIE)** —
architecturally a TIE too, but one that adds two extra jobs on top of plain
protocol/format conversion:

- **[Canonical Data Model](https://www.enterpriseintegrationpatterns.com/patterns/messaging/CanonicalDataModel.html)**
  — the NW HL7 v2 standard itself. Every Trust's TIE converts its own local flavour
  *into* this one shared shape on the way in, and *out of* it again on the way back —
  so the RIE only ever has to understand one message shape, not one per Trust × one per
  LIMS.
- **[Message Routing](https://www.enterpriseintegrationpatterns.com/patterns/messaging/MessageRoutingIntro.html)**
  — the RIE decides *which* LIMS a given order goes to (and converts the canonical
  message into that LIMS's own required flavour on the way out), and which Trusts/Shared
  Care Records a given report fans out to on the way back.

The payoff of doing both together: an Order Placer (an NHS Trust's EPR) never needs to
know or care whether its order ends up at Histotrac, StarLIMS, or iGene — and NW
Genomics' own systems never need to know or care which EPR flavour a given Trust
happens to run. Both sides only ever deal with the one canonical shape; the RIE hides
everything either side of it.

This notebook plays the TIE's part in that picture by hand, in plain Python — the
`v2 → v2` equivalent of what `09-genomic-order-management-fhir-to-hl7v2-for-lims.ipynb`
did for `FHIR → v2`: no API call, just field-by-field mapping, so the conversion logic
is visible rather than hidden behind a tool call.

In [1]:
import subprocess


def read_segments(path):
    with open(path, newline="") as f:
        raw = f.read()
    return [s for s in raw.replace("\r\n", "\r").split("\r") if s]


def print_segments(path):
    segments = read_segments(path)
    for segment in segments:
        print(segment)
    return segments


def by_segment_type(segments):
    grouped = {}
    for segment in segments:
        fields = segment.split("|")
        grouped.setdefault(fields[0], []).append(fields)
    return grouped

## Step 1: the order as it reaches the local lab today

`Clatterbridge-Order.txt` — an `ORM^O01` order, Clatterbridge's own Meditech-originated
flavour, addressed straight at the local lab (`MSH-3`/`MSH-5` = `LAB`/`CCC`). This is
the "before" state the H&I use case's migration is replacing: no TIE/RIE/canonical
model in the loop at all, just Meditech talking directly to whatever the lab's own
system expects.

In [2]:
clatterbridge_segments = print_segments("Input/V2/O01/Clatterbridge-Order.txt")

MSH|^~\&|LAB|CCC||CCC|20260109164121+0000||ORM^O01|27076860.1|P|2.4|||AL|NE|
PID|1||CB07442115^^^REN^MR~9737383206^^^NHS^NH|C2-B20250217143952054|LIVERPOOL^Ned||19420618|M||M|^HUYTON^LIVERPOOL^MERSEYSIDE^L7 8XP|||||M||AN0000281729||||||||||||||01|
PV1|1|O|PAL_C863^^^REN||||C6167060^Wells^Matthew^^^^^^XX|||CLIN HAEM||||||||RCR||NHS|||||||||||||||||||CCC||REG|||202601081200|
ORC|NW|5595441^LAB||090126:RI6||N|^^^^^R||202601091641||||||||||||The Clatterbridge Cancer Centre NHS Foundation Trust^^REN^^^ODS
NTE|1||Patient Location 5.Ward 5 TYA|
OBR|1|5595441^LAB||STR^Chimerism (CCC-L)^L|||202601091638||||||||EB|C6167060^Wells^Matthew^^^^^^XX||05094286||||||LAB|||^^^^^R|
OBR|2|5595441^LAB||CD348^Lymph Sub (CD3/4/8/45) (CCC-L)^L|||202601091638||||||||S|C6167060^Wells^Matthew^^^^^^XX||05094286||||||LAB|||^^^^^R|


[`Input/V2/O01/Clatterbridge-Order-review.md`](https://github.com/nw-gmsa/Testing/blob/main/Input/V2/O01/Clatterbridge-Order-review.md)
reviews this exact message against the NW-GMSA `OML_O21` spec in detail. Most of that
gap list is about `OML_O21` structure specifically, which this use case doesn't
actually target (H&I uses `ORM_O01`, see above). This notebook picks up four fields
from that review that are about **identifiers and addressing**, not message structure,
and are exactly the kind of thing a TIE fixes on every message it converts:

| Field | Current value | Problem |
| --- | --- | --- |
| `MSH-3`/`MSH-4`/`MSH-5`/`MSH-6` (Sending/Receiving Application/Facility) | `LAB`/`CCC`/*(empty)*/`CCC` | `LAB` and `CCC` are Clatterbridge's own local shorthand, not ODS codes — `CCC` isn't Clatterbridge's actual ODS code (`REN` is) at all. |
| `PID-3` (MRN repeat) | `CB07442115^^^REN^MR` | Nothing — this is already the target shape, and the model for the identifier fields below. |
| `ORC-2` / `OBR-2` (Placer Order Number) | `5595441^LAB` | `LAB` is a local system name, not Clatterbridge's ODS code (`REN`) — no national identifier authority is expressed. |
| `PID-18` (Patient Account Number) | `AN0000281729` | Right *value*, wrong *field* — NW-standard messages carry this in `PV1`, not `PID-18` (see below), and it has no CX typing at all here. |


### `MSH`: a generic application, a real facility

`MSH-3`/`MSH-5` (Sending/Receiving Application) and `MSH-4`/`MSH-6` (Sending/Receiving
Facility) are two different kinds of thing, and the NW-standard order this notebook
builds treats them differently:

- **Facility is always a real ODS code** — the sending side is Clatterbridge itself
  (`REN`), and the receiving side, at this point in the flow, is NW Genomics as a whole
  (`699X0`, the same GLH ODS code
  `09-genomic-order-management-fhir-to-hl7v2-for-lims.ipynb` uses as `NW_GLH_ODS`) —
  not yet Histotrac specifically.
- **Application is a generic role name, not a product name.** Clatterbridge's TIE
  doesn't address the message to "Histotrac" — it doesn't know, and doesn't need to
  know, which LIMS the RIE will eventually route it to. It sends to a generic `LIMS`.
  Symmetrically, on the way back, NW Genomics doesn't address a report to "Meditech" —
  it sends to a generic `EPR`. *Which* specific LIMS or EPR product is actually on the
  other end is something only the RIE (which LIMS) or the TIE (which EPR) needs to
  know — that's exactly the [Message
  Routing](https://www.enterpriseintegrationpatterns.com/patterns/messaging/MessageRoutingIntro.html)
  step from the diagram above, and it's why this boundary message can't and shouldn't
  name a product.

| Direction | Sending App | Sending Facility | Receiving App | Receiving Facility |
| --- | --- | --- | --- | --- |
| Order, Trust → NW | `EPR` | `REN` (Clatterbridge) | `LIMS` | `699X0` (NW Genomics) |
| Report, NW → Trust | `LIMS` | `699X0` (NW Genomics) | `EPR` | `REN` (Clatterbridge) |


In [3]:
NW_GLH_ODS = "699X0"  # same constant as notebook 09's NW_GLH_ODS
CLATTERBRIDGE_ODS = "REN"

msh = by_segment_type(clatterbridge_segments)["MSH"][0]
print("before: MSH-3/4/5/6 =", msh[2], "/", msh[3], "/", repr(msh[4]), "/", msh[5])

new_msh_sending_app, new_msh_sending_facility = "EPR", CLATTERBRIDGE_ODS
new_msh_receiving_app, new_msh_receiving_facility = "LIMS", NW_GLH_ODS
print("after:  MSH-3/4/5/6 =", new_msh_sending_app, "/", new_msh_sending_facility,
      "/", new_msh_receiving_app, "/", new_msh_receiving_facility)

before: MSH-3/4/5/6 = LAB / CCC / '' / CCC
after:  MSH-3/4/5/6 = EPR / REN / LIMS / 699X0


### The identifier shapes

Two HL7 v2 data types are doing the work in the remaining three fields:

- **`CX`** (Extended Composite ID) — `<ID>^<check digit>^<check digit scheme>
  ^<assigning authority>^<identifier type code>`. `PID-3`'s MRN repeat already uses
  this correctly: `CB07442115^^^REN^MR` — ID `CB07442115`, assigning authority `REN`
  (Clatterbridge's ODS code), identifier type `MR` (Medical Record Number, HL7 table
  `0203`). This is the pattern the Account Number needs too, with identifier type `AN`
  (Account Number) instead of `MR`.
- **`EI`** (Entity Identifier) — `<entity identifier>^<namespace ID>^<universal ID>
  ^<universal ID type>`, used by `ORC-2`/`OBR-2` (Placer Order Number). `5595441^LAB`
  only populates the first two components — the *namespace* (`LAB`, a local system
  name) but not a *universal ID* an outside system could resolve. Adding `REN` as the
  universal ID and `ODS` as its type (the same "ODS" typing already used a few fields
  along, in `ORC-21`'s `...^^REN^^^ODS`) gives it exactly what `PID-3`'s MRN already
  has, without discarding the original local namespace.

In [4]:
def cx(value, assigning_authority, id_type):
    """CX: <ID>^<check digit>^<check digit scheme>^<assigning authority>^<identifier type>"""
    return f"{value}^^^{assigning_authority}^{id_type}"


def ei(value, namespace_id, universal_id, universal_id_type):
    """EI: <entity identifier>^<namespace ID>^<universal ID>^<universal ID type>"""
    return f"{value}^{namespace_id}^{universal_id}^{universal_id_type}"


print(cx("CB07442115", CLATTERBRIDGE_ODS, "MR"), " <- PID-3's MRN, already this shape")

CB07442115^^^REN^MR  <- PID-3's MRN, already this shape


### Fixing the Placer Order Number

`ORC-2` and both `OBR-2`s carry the same value, `5595441^LAB` — add the ODS-typed
universal ID, keep the existing namespace.

In [5]:
by_seg = by_segment_type(clatterbridge_segments)

placer_order_value, placer_order_namespace = by_seg["ORC"][0][2].split("^")
new_placer_order_number = ei(placer_order_value, placer_order_namespace, CLATTERBRIDGE_ODS, "ODS")

print("before:", by_seg["ORC"][0][2])
print("after: ", new_placer_order_number)

before: 5595441^LAB
after:  5595441^LAB^REN^ODS


### Moving the Account Number into `PV1`

`PID-18` (Patient Account Number) holds `AN0000281729` — a real value in the wrong
field for a NW-standard message. `histotrac-MFT.txt`, a real order from a different
Trust already in the target shape, confirms where it belongs: its `PV1-19` (Visit
Number) is the only place anything resembling a visit/account identifier appears —
`PID-18` in that file is empty, and the same slot (`PV1-19`) is echoed in `ORC-4`
(Placer Group Number) too.

In [6]:
histotrac_order_segments = read_segments("Input/V2/O01/histotrac-MFT.txt")
histotrac_by_seg = by_segment_type(histotrac_order_segments)

print("histotrac-MFT.txt PID-18 (Patient Account Number):", repr(histotrac_by_seg["PID"][0][18]))
print("histotrac-MFT.txt PV1-19 (Visit Number):          ", repr(histotrac_by_seg["PV1"][0][19]))
print("histotrac-MFT.txt ORC-4  (Placer Group Number):   ", repr(histotrac_by_seg["ORC"][0][4]))

histotrac-MFT.txt PID-18 (Patient Account Number): ''
histotrac-MFT.txt PV1-19 (Visit Number):           'visit_no'
histotrac-MFT.txt ORC-4  (Placer Group Number):    'visit_no'


In [7]:
account_number = by_seg["PID"][0][18]
new_account_number_field = cx(account_number, CLATTERBRIDGE_ODS, "AN")

print("PID-18 before:", repr(account_number))
print("PV1-19 after: ", repr(new_account_number_field))

PID-18 before: 'AN0000281729'
PV1-19 after:  'AN0000281729^^^REN^AN'


### Assembling the converted message

Four edits, applied to copies of the original segments: set `MSH-3`/`4`/`5`/`6` to the
generic-application/real-ODS-facility shape, clear `PID-18`, set `PV1-19` to the
newly-typed account number, and give `ORC-2` and both `OBR-2`s the ODS-typed Placer
Order Number. Everything else — including the structural gaps
`Clatterbridge-Order-review.md` flags but this notebook isn't fixing (one `ORC` shared
across two `OBR`s, the missing `SPM`, `OBR-31`) — is carried over unchanged, since
those are `OML_O21`-structure points that don't apply to this use case's `ORM_O01`
shape.

In [8]:
converted_segments = []
for segment in clatterbridge_segments:
    fields = segment.split("|")

    if fields[0] == "MSH":
        fields[2] = new_msh_sending_app
        fields[3] = new_msh_sending_facility
        fields[4] = new_msh_receiving_app
        fields[5] = new_msh_receiving_facility
    elif fields[0] == "PID":
        fields[18] = ""
    elif fields[0] == "PV1":
        fields[19] = new_account_number_field
    elif fields[0] in ("ORC", "OBR"):
        value, namespace = fields[2].split("^")
        fields[2] = ei(value, namespace, CLATTERBRIDGE_ODS, "ODS")

    converted_segments.append("|".join(fields))

converted_message = "\r".join(converted_segments) + "\r"
print(converted_message.replace("\r", "\r\n"))

MSH|^~\&|EPR|REN|LIMS|699X0|20260109164121+0000||ORM^O01|27076860.1|P|2.4|||AL|NE|
PID|1||CB07442115^^^REN^MR~9737383206^^^NHS^NH|C2-B20250217143952054|LIVERPOOL^Ned||19420618|M||M|^HUYTON^LIVERPOOL^MERSEYSIDE^L7 8XP|||||M||||||||||||||||01|
PV1|1|O|PAL_C863^^^REN||||C6167060^Wells^Matthew^^^^^^XX|||CLIN HAEM||||||||RCR|AN0000281729^^^REN^AN|NHS|||||||||||||||||||CCC||REG|||202601081200|
ORC|NW|5595441^LAB^REN^ODS||090126:RI6||N|^^^^^R||202601091641||||||||||||The Clatterbridge Cancer Centre NHS Foundation Trust^^REN^^^ODS
NTE|1||Patient Location 5.Ward 5 TYA|
OBR|1|5595441^LAB^REN^ODS||STR^Chimerism (CCC-L)^L|||202601091638||||||||EB|C6167060^Wells^Matthew^^^^^^XX||05094286||||||LAB|||^^^^^R|
OBR|2|5595441^LAB^REN^ODS||CD348^Lymph Sub (CD3/4/8/45) (CCC-L)^L|||202601091638||||||||S|C6167060^Wells^Matthew^^^^^^XX||05094286||||||LAB|||^^^^^R|



In [9]:
with open("Output/V2/O01/Clatterbridge-Order-NWstandard.txt", "w", newline="") as f:
    f.write(converted_message)

for original, converted in zip(clatterbridge_segments, converted_segments):
    if original != converted:
        print("- " + original)
        print("+ " + converted)
        print()

- MSH|^~\&|LAB|CCC||CCC|20260109164121+0000||ORM^O01|27076860.1|P|2.4|||AL|NE|
+ MSH|^~\&|EPR|REN|LIMS|699X0|20260109164121+0000||ORM^O01|27076860.1|P|2.4|||AL|NE|

- PID|1||CB07442115^^^REN^MR~9737383206^^^NHS^NH|C2-B20250217143952054|LIVERPOOL^Ned||19420618|M||M|^HUYTON^LIVERPOOL^MERSEYSIDE^L7 8XP|||||M||AN0000281729||||||||||||||01|
+ PID|1||CB07442115^^^REN^MR~9737383206^^^NHS^NH|C2-B20250217143952054|LIVERPOOL^Ned||19420618|M||M|^HUYTON^LIVERPOOL^MERSEYSIDE^L7 8XP|||||M||||||||||||||||01|

- PV1|1|O|PAL_C863^^^REN||||C6167060^Wells^Matthew^^^^^^XX|||CLIN HAEM||||||||RCR||NHS|||||||||||||||||||CCC||REG|||202601081200|
+ PV1|1|O|PAL_C863^^^REN||||C6167060^Wells^Matthew^^^^^^XX|||CLIN HAEM||||||||RCR|AN0000281729^^^REN^AN|NHS|||||||||||||||||||CCC||REG|||202601081200|

- ORC|NW|5595441^LAB||090126:RI6||N|^^^^^R||202601091641||||||||||||The Clatterbridge Cancer Centre NHS Foundation Trust^^REN^^^ODS
+ ORC|NW|5595441^LAB^REN^ODS||090126:RI6||N|^^^^^R||202601091641||||||||||||The Clatte

## Step 2: checking the result against a real NW-standard order

`histotrac-MFT.txt` is a different real Trust (Manchester University NHS Foundation
Trust's Epic/Beaker) sending the same kind of order, already converted by that Trust's
own TIE - but its `MSH-5`/`MSH-6` (`Histotrac`/`HISTOTRAC`) name the LIMS directly,
rather than the generic `LIMS`/`699X0` shape Step 1 just built. That's not a
contradiction: this file represents a *later* stage in the diagram above - after the
RIE has already done its own routing/conversion step and re-addressed the message to
the specific LIMS - not the TIE-to-RIE handoff Step 1 is modelling. It also isn't a
perfect model of that later stage either, worth naming rather than hiding: its own
`ORC-2` (`300995243^EPC`) uses `EPC`, Epic's own internal system code, not MFT's ODS
code (`R0A`, visible instead in `PID-3`'s MRN — `R0A4990415^^^R0A^MR`). So even a real,
already-flowing message doesn't ODS-type every identifier consistently — this
notebook's own conversion above is deliberately stricter than either source file,
applying the same `PID-3`-MRN pattern everywhere rather than only where a given Trust's
feed happens to have already done it.

In [10]:
for label, path in [
    ("Clatterbridge-Order.txt", "Input/V2/O01/Clatterbridge-Order.txt"),
    ("histotrac-MFT.txt", "Input/V2/O01/histotrac-MFT.txt"),
]:
    with open(path, newline="") as f:
        raw = f.read()
    msh = [s for s in raw.replace("\r\n", "\r").split("\r") if s][0].split("|")
    print(f"{label:26} MSH-9 = {msh[8]:16} MSH-12 = {msh[11]}")

Clatterbridge-Order.txt    MSH-9 = ORM^O01          MSH-12 = 2.4
histotrac-MFT.txt          MSH-9 = ORM^O01          MSH-12 = 2.5.1


## Step 3: the report coming back the other way

Reports flow Histotrac → RIE → Trust the same way orders flow Trust → RIE → LIMS, just
reversed. `histotrac-MFT.txt` (R01) and `histotrac.txt` both follow the NW-GMSA IG's
own PDF-report pattern for this discipline
([nw-gmsa.github.io/en/hl7v2.html](https://nw-gmsa.github.io/en/hl7v2.html)): `OBR-4`
carries the SNOMED CT discipline code
`909871000000100^Histocompatibility and immunogenetics^SNM3`, and the report itself
travels as a single `OBX-5` `ED` (encapsulated data) field — `MOL^IM^PDF^Base64^<data>`,
`OBX-11` `F` (final). Unlike the order side, neither file has an account-number-shaped
gap to fix — `PID-18` is empty and unused in both, matching the convention Step 2 just
confirmed.

In [11]:
histotrac_report_segments = read_segments("Input/V2/R01/histotrac-MFT.txt")
report_by_seg = by_segment_type(histotrac_report_segments)

obr = report_by_seg["OBR"][0]
obx = report_by_seg["OBX"][0]

print("OBR-4 (discipline code):", obr[4])
print("OBX-2 (value type):     ", obx[2])
print("OBX-11 (result status): ", obx[11])
print("OBX-5 length (base64):  ", len(obx[5]), "chars")
print("PID-18 (Account Number):", repr(report_by_seg["PID"][0][18]))

OBR-4 (discipline code): 909871000000100^Histocompatibility and immunogenetics^SNM3
OBX-2 (value type):      ED
OBX-11 (result status):  F
OBX-5 length (base64):   1966 chars
PID-18 (Account Number): ''


## Step 4: the same fixtures, run live end-to-end

`IntegrationTest.py`'s `clatterbridge_histotrac` group already exercises all of this -
these two `O01` orders plus three `R01` reports (`histotrac-MFT.txt`, `histotrac.txt`,
and `Clatterbridge-REN-ORU_R01.txt`, the report leg back to Clatterbridge that this
notebook didn't walk through by hand) - against the live RIE: `transformToFHIR` → send
to `V2_SERVER` → ACK check → `transformToV2`, the same three stages every group in this
repo runs. Requires a live `V2_TOOLS`/`V2_SERVER` (see `.env`); this cell isn't part of
the hand-built conversion above, just a way to confirm the real engine agrees with it.

In [ ]:
result = subprocess.run(
    ["python", "IntegrationTest.py", "--group", "clatterbridge_histotrac"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

## Summary

- The NW HL7 v2 standard is a **union**: NHS England's ADT spec for `MSH`/`PID`/`PD1`/
  `PV1`, DHCW's `ORU_R01` IG for `ORC`/`OBR`/`OBX`/`SPM`, and NHS England's Data Model
  and Dictionary for the identifiers and coded values that run through both.
- For H&I specifically, "NW standard" means `ORM_O01`/`ORU_R01` at `MSH-12 = 2.5.1` —
  not `OML_O21`, which is the only order profile `hl7v2.html` formally documents.
- Converting a Trust's local flavour into that standard is, concretely, identifier and
  addressing work:
  - `MSH`'s Sending/Receiving Application fields are a **generic role name** (`EPR`,
    `LIMS`) — this boundary doesn't know, and doesn't need to know, the specific
    product on the other end; only the RIE or TIE that routes the message next does.
  - `MSH`'s Sending/Receiving Facility fields, and every organisation-scoped
    identifier's assigning authority (`MRN`, Placer Order Number, Account Number),
    carry the actual ODS code — using each field's own data type (`CX` for
    `PID-3`/`PID-18`/`PV1-19`, `EI` for `ORC-2`/`OBR-2`).
  - Each value goes in the field the standard actually defines for it, even when the
    source system (here, `PID-18`) puts it somewhere else.
- That per-Trust conversion is what lets NW Genomics' RIE sit in the middle as a
  [Canonical Data Model](https://www.enterpriseintegrationpatterns.com/patterns/messaging/CanonicalDataModel.html)
  plus [Message Router](https://www.enterpriseintegrationpatterns.com/patterns/messaging/MessageRoutingIntro.html)
  without knowing about every Trust × every LIMS combination directly — Histotrac is
  one destination among several (StarLIMS, iGene are the others already live in this
  repo's other fixtures/notebooks), and the same TIE → RIE → LIMS pattern, plus its
  mirror image for reports, applies regardless of which LIMS or which Trust sits at
  either end. As the H&I use case page puts it: *"The Data Contract only exists between
  NHS Trusts and NW Genomics — it does not apply to local integrations with EPR or LIMS
  systems."*